<a href="https://colab.research.google.com/github/syedmahmoodiagents/NLP/blob/main/Full_Many_One_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
corpus = [
    "I love doing mathematics",
    "calculus is great mathematics",
    "mathematics is quite useful"
]

In [ ]:
import spacy
import torch
import torch.nn as nn
# from torch.utils.data import Dataset, DataLoader

### Prepartion of data

In [ ]:
nlp = spacy.load("en_core_web_sm")

In [ ]:
word2idx = {"<PAD>":0, "<UNK>":1}
idx2word = {0:"<PAD>", 1:"<UNK>"}

index = 2
for sentence in corpus:
    doc = nlp(sentence)
    for token in doc:
        word = token.text.lower()
        if word not in word2idx:
            word2idx[word] = index
            idx2word[index] = word
            index += 1


In [ ]:
print(word2idx)

{'<PAD>': 0, '<UNK>': 1, 'i': 2, 'love': 3, 'doing': 4, 'mathematics': 5, 'calculus': 6, 'is': 7, 'great': 8, 'quite': 9, 'useful': 10}


In [ ]:
vocab_size = len(word2idx)

In [ ]:
vocab_size

11

In [ ]:
encoded = []
for sentence in corpus:
    doc = nlp(sentence)
    ids = []
    for token in doc:
        ids.append(word2idx[token.text.lower()])
    encoded.append(ids)

In [ ]:
print(encoded)

[[2, 3, 4, 5], [6, 7, 8, 5], [5, 7, 9, 10]]


### creating many to one samples

In [ ]:
X = []
Y = []

for sent in encoded:
    for i in range(1, len(sent)):
        X.append(sent[:i]) # starting from 0 less than i
        Y.append(sent[i]) # upto i

In [ ]:
X

[[2], [2, 3], [2, 3, 4], [6], [6, 7], [6, 7, 8], [5], [5, 7], [5, 7, 9]]

In [ ]:
Y

[3, 4, 5, 7, 8, 5, 7, 9, 10]

In [ ]:
for x, y in zip(X, Y):
    print(x,"=>", y)

[2] => 3
[2, 3] => 4
[2, 3, 4] => 5
[6] => 7
[6, 7] => 8
[6, 7, 8] => 5
[5] => 7
[5, 7] => 9
[5, 7, 9] => 10


### Padding

In [ ]:
max_len = max(len(x) for x in X)

In [ ]:
max_len

3

In [ ]:
3*[0]

[0, 0, 0]

In [ ]:
[0] * (max_len - 1)

[0, 0]

In [ ]:
for i in range(len(X)):
    print(X[i])

[2]
[2, 3]
[2, 3, 4]
[6]
[6, 7]
[6, 7, 8]
[5]
[5, 7]
[5, 7, 9]


In [ ]:
for i in range(len(X)):
    pad = [0] * (max_len - len(X[i]))
    X[i] = pad + X[i]

In [ ]:
X

[[0, 0, 2],
 [0, 2, 3],
 [2, 3, 4],
 [0, 0, 6],
 [0, 6, 7],
 [6, 7, 8],
 [0, 0, 5],
 [0, 5, 7],
 [5, 7, 9]]

In [ ]:
X = torch.tensor(X, dtype=torch.long)
Y = torch.tensor(Y, dtype=torch.long)

print(X.shape)
print(Y.shape)

torch.Size([9, 3])
torch.Size([9])


### model network

In [ ]:
class NextWordRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(input_size=embedding_dim, hidden_size=hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        # (batch, seq_len)
        x = self.embedding(x)
        output, hidden = self.rnn(x)
        last_output = output[:, -1, :] # Last hidden state
        out = self.fc(last_output)
        return out


In [ ]:
model = NextWordRNN(vocab_size=vocab_size, embedding_dim=16, hidden_dim=32)

In [ ]:
criterion = nn.CrossEntropyLoss() # built in Softmax() # always for classification
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

### Training

In [ ]:

for epoch in range(300):
    optimizer.zero_grad()
    predictions = model(X)
    loss = criterion(predictions, Y)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d} Loss = {loss.item():.4f}")

Epoch  20 Loss = 0.0706
Epoch  40 Loss = 0.0069
Epoch  60 Loss = 0.0034
Epoch  80 Loss = 0.0025
Epoch 100 Loss = 0.0020
Epoch 120 Loss = 0.0017
Epoch 140 Loss = 0.0014
Epoch 160 Loss = 0.0012
Epoch 180 Loss = 0.0011
Epoch 200 Loss = 0.0009
Epoch 220 Loss = 0.0008
Epoch 240 Loss = 0.0008
Epoch 260 Loss = 0.0007
Epoch 280 Loss = 0.0006
Epoch 300 Loss = 0.0006


### Inference

In [ ]:
def convert_to_ids(text):
    doc = nlp(text)
    ids = []
    for token in doc:
        word = token.text.lower()
        ids.append(word2idx.get(word, word2idx["<UNK>"]))

    if len(ids) < max_len:
        ids = [0] * (max_len - len(ids)) + ids
    else:
        # print("==>")
        ids = ids[-max_len:]

    return ids

In [ ]:
convert_to_ids("is quite useful")

[7, 9, 10]

In [ ]:
def prediction(ids):
    ids = torch.tensor([ids], dtype=torch.long)
    prediction = model(ids)
    predicted_index = torch.argmax(prediction, dim=1).item()
    return idx2word[predicted_index]

In [ ]:
prediction(convert_to_ids("I love"))

'doing'

### Complete Function

In [ ]:
def predict_next_word(text):
    doc = nlp(text)
    seq = []
    for token in doc:
        word = token.text.lower()
        seq.append(word2idx.get(word, word2idx["<UNK>"]))

    if len(seq) < max_len:
        seq = [0] * (max_len - len(seq)) + seq
    else:
        seq = seq[-max_len:]

    seq = torch.tensor([seq], dtype=torch.long)

    model.eval()
    with torch.no_grad():
        prediction = model(seq)
        predicted_index = torch.argmax(prediction, dim=1).item()

    return idx2word[predicted_index]

In [ ]:
predict_next_word("I love doing")

'mathematics'